<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.2-iam-security/notebooks/GCP_Capstone_1.2_IAM_Security.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.2 IAM & Security for GenAI Projects
**Netsetos GenAI Engineering — GCP Capstone**

Hands-on: Create dedicated SAs, store secrets, verify ADC, query audit logs.


## Cell 1: Authenticate & Set Project


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
!gcloud config set project {PROJECT_ID}


## Cell 2: Inspect the Default Service Account


In [ ]:
# Find the default compute SA and its dangerous Editor role
!gcloud iam service-accounts list --format='table(email,displayName)'
print('\n--- Roles on default SA ---')
!gcloud projects get-iam-policy {PROJECT_ID} \
  --flatten='bindings[].members' \
  --filter='bindings.members:compute@developer' \
  --format='table(bindings.role)'


## Cell 3: Create Dedicated Service Account


In [ ]:
# Create app SA with least-privilege intent
!gcloud iam service-accounts create sa-documind-app \
  --display-name='DocuMind Application SA' \
  --description='Runs Gemini chatbot on Cloud Run'

# Verify
!gcloud iam service-accounts list --filter='email:sa-documind' --format='table(email,displayName)'


## Cell 4: Grant Exactly 4 IAM Roles


In [ ]:
SA = f'sa-documind-app@{PROJECT_ID}.iam.gserviceaccount.com'

roles = [
    'roles/aiplatform.user',
    'roles/datastore.user', 
    'roles/secretmanager.secretAccessor',
    'roles/storage.objectViewer'
]

for role in roles:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
      --member='serviceAccount:{SA}' --role='{role}' --quiet
    print(f'  Granted: {role}')

# Verify
print('\n--- Roles on sa-documind-app ---')
!gcloud projects get-iam-policy {PROJECT_ID} \
  --flatten='bindings[].members' \
  --filter='bindings.members:sa-documind-app' \
  --format='table(bindings.role)'


## Cell 5: Create Secrets in Secret Manager


In [ ]:
# Create two secrets
!echo -n 'test-gemini-key-12345' | gcloud secrets create gemini-config --data-file=- --replication-policy=automatic --labels='env=dev'
!echo -n 'super-secure-db-pass' | gcloud secrets create db-password --data-file=- --replication-policy=automatic

# List secrets
!gcloud secrets list --format='table(name,labels)'


## Cell 6: Access Secrets from Python


In [ ]:
from google.cloud import secretmanager

def get_secret(secret_id, version='latest'):
    client = secretmanager.SecretManagerServiceClient()
    name = f'projects/{PROJECT_ID}/secrets/{secret_id}/versions/{version}'
    response = client.access_secret_version(request={'name': name})
    return response.payload.data.decode('UTF-8')

# Test both secrets
for sid in ['gemini-config', 'db-password']:
    val = get_secret(sid)
    print(f'  {sid}: {val[:4]}****')


## Cell 7: Test Gemini Call with Proper SA


In [ ]:
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location='us-central1')
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Explain what IAM means in cloud security in 2 sentences.'
)
print('Response:', response.text)
print(f'Tokens: {response.usage_metadata.total_token_count}')


## Cell 8: Query Audit Logs


In [ ]:
# See who made API calls recently
!gcloud logging read \
  'logName:"cloudaudit.googleapis.com"' \
  --project={PROJECT_ID} --freshness=1d --limit=5 \
  --format='table(timestamp,protoPayload.methodName,protoPayload.authenticationInfo.principalEmail)'


## Cell 9: Security Audit Script


In [ ]:
import subprocess, json

result = subprocess.run(
    ['gcloud', 'projects', 'get-iam-policy', PROJECT_ID, '--format=json'],
    capture_output=True, text=True
)
policy = json.loads(result.stdout)

DANGEROUS = {'roles/editor', 'roles/owner'}
sa_roles = {}

for binding in policy.get('bindings', []):
    role = binding['role']
    for member in binding.get('members', []):
        if 'serviceAccount:' in member:
            sa = member.split(':')[1]
            sa_roles.setdefault(sa, []).append(role)

print('\n Security Audit Report')
print('=' * 60)
alerts = 0
for sa, roles in sorted(sa_roles.items()):
    has_danger = any(r in DANGEROUS for r in roles)
    icon = '🔴' if has_danger else '🟢'
    print(f'\n{icon} {sa}')
    for r in roles:
        flag = ' ⚠ OVERPRIVILEGED' if r in DANGEROUS else ''
        print(f'   {r}{flag}')
    if has_danger:
        alerts += 1

print(f'\n Summary: {len(sa_roles)} SAs, {alerts} with dangerous roles')
if alerts:
    print(' ACTION REQUIRED: Remove Editor/Owner from service accounts!')
else:
    print(' All service accounts follow least privilege.')


## ✅ Security Setup Complete!

- ✅ Inspected dangerous default SA
- ✅ Created dedicated sa-documind-app with 4 roles
- ✅ Stored secrets in Secret Manager
- ✅ Accessed secrets from Python
- ✅ Queried audit logs
- ✅ Built security audit script

**Next: Lesson 1.3 — Your First Gemini Call (Deep Dive)**
